# CMap発現プロファイルからtarget-associated latent factorを探す

このデモでは connectivity map project の化合物処理後の遺伝子発現データを解析する。

1. セットアップ
2. データサイズとheadの確認
3. 着目化合物に似た発現プロファイルを示す化合物の確認
4. DBSCANによるクラスタリングとt-SNE可視化
5. PCA初期抽出 + varimax回転による潜在因子抽出
6. target-high factor の上位・下位 sample の確認
7. 因子数の妥当性の確認


## 0. 最初に

まず **[ドライブにコピー]** を押してください。
その後, このノートブックを上から順に実行してください。

In [ ]:
#@title 1. セットアップ { display-mode: "form" }

install_mode = "github"  #@param ["github", "upload_zip", "skip"]
GITHUB_OWNER = "mizuno-group"  #@param {type:"string"}
REPO_NAME = "latent-demo"  #@param {type:"string"}
GIT_REF = "main"  #@param {type:"string"}  # branch, tag, or commit hash

import os
import sys
import inspect
import subprocess
import matplotlib.pyplot as plt

if install_mode != "skip":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
    ])
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "latent-demo"], check=False)

    if install_mode == "github":
        package_spec = f"git+https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git@{GIT_REF}"
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--no-cache-dir", "--force-reinstall", "--no-deps", package_spec,
        ])
    elif install_mode == "upload_zip":
        from google.colab import files
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.endswith(".zip")]
        if not zip_names:
            raise RuntimeError("zipファイルをアップロードしてください。")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--no-cache-dir", "--force-reinstall", "--no-deps", zip_names[0],
        ])

for name in list(sys.modules):
    if name == "latent_demo" or name.startswith("latent_demo."):
        del sys.modules[name]

import latent_demo
print("latent_demo version:", getattr(latent_demo, "__version__", "unknown"))
print("latent_demo path:", os.path.dirname(inspect.getfile(latent_demo)))

from latent_demo.cmap import (
    prepare_cmap_data,
    compute_compound_correlation,
    rank_target_correlations,
    cluster_compounds_dbscan,
    fit_cmap_varimax_factors,
    compare_cmap_factor_numbers,
    plot_dbscan_tsne_scatter,
    plot_ranked_factor_scores,
    plot_cmap_factor_number_summary,
    ESTROGEN_TERMS,
    ANTI_ESTROGEN_TERMS,
    match_samples,
    summarize_terms,
)

print("Setup finished.")


## 1. 実習設定

まずはデフォルト値のまま実行してください。


In [ ]:
#@title 2. パラメータを選ぶ { display-mode: "form" }
n_top_genes = 3000  #@param {type:"slider", min:500, max:6000, step:500}
n_components = 40  #@param {type:"slider", min:5, max:80, step:5}
target_sample = "estradiol"  #@param {type:"string"}
top_n = 20  #@param {type:"slider", min:5, max:40, step:5}
dbscan_eps = 0.6  #@param {type:"slider", min:0.3, max:0.9, step:0.05}
dbscan_min_samples = 3  #@param {type:"slider", min:2, max:8, step:1}
tsne_perplexity = 30  #@param {type:"slider", min:5, max:60, step:5}


## 2. データを読み込む

repo内に同梱された `ref_cmap.csv` を読み込む。  
オミクス解析の慣例に従って, 行は遺伝子, 列は化合物/sampleとしている。  
※一般の統計・機械学習モデルでは行がsampleであることが多いものの, オミクスデータはlarge p small nの形状が一般的。

In [ ]:
cmap = prepare_cmap_data(n_top_genes=n_top_genes)

print("genes used   :", cmap.gene_by_compound.shape[0])
print("compounds    :", cmap.gene_by_compound.shape[1])
print("target exists:", target_sample in cmap.compound_by_gene.index)


## 3. データサイズと head を見る

まずデータを確認する。

In [ ]:
display(cmap.gene_by_compound.iloc[:5, :8])

print("compound x gene matrix:", cmap.compound_by_gene.shape)
display(cmap.compound_by_gene.iloc[:5, :8])


## 4. 着目化合物に似ているものを探す

化合物ごとの発現プロファイルが似ていれば化合物間相関が高くなる。  
ここでは`target_sample` と相関が高い化合物をリストとして確認する。


In [ ]:
correlation = compute_compound_correlation(cmap.compound_by_gene)
target_correlations = rank_target_correlations(correlation, target_sample=target_sample)

print(f"Most similar to {target_sample}")
display(target_correlations.head(top_n).round(3))

print(f"Most opposite to {target_sample}")
display(target_correlations.tail(top_n).sort_values("correlation").round(3))


**tips**：`target_sample` を `tamoxifen` や `dexamethasone` などに変えると別の薬理応答の近さを確認できる。


## 5. DBSCANで化合物をクラスタリングしてt-SNEで可視化する

相関距離 `1 - correlation` を使い, 近い化合物のクラスターをDBSCANで確認する。  
可視化には同じ相関距離に基づくt-SNEを用いる。  
※umapはセットアップの手間を考慮して今回使用しない。

In [ ]:
clusters = cluster_compounds_dbscan(
    correlation,
    eps=dbscan_eps,
    min_samples=dbscan_min_samples,
)

cluster_summary = (
    clusters["cluster"].value_counts().sort_index()
    .rename_axis("cluster")
    .to_frame("n_samples")
)
display(cluster_summary)

highlight_terms = [target_sample, "estradiol", "estrone", "estriol", "tamoxifen", "raloxifene", "fulvestrant", "clomifene"]
plot_dbscan_tsne_scatter(
    correlation,
    clusters,
    perplexity=tsne_perplexity,
    highlight_terms=highlight_terms,
    title=f"DBSCAN on correlation distance with t-SNE (eps={dbscan_eps}, min_samples={dbscan_min_samples}, perplexity={tsne_perplexity})",
)
plt.show()


# summary
estrogen_hits = match_samples(cmap.compound_by_gene.index, ESTROGEN_TERMS)
anti_hits = match_samples(cmap.compound_by_gene.index, ANTI_ESTROGEN_TERMS)

selected_names = []
for name in [target_sample] + estrogen_hits + anti_hits:
    if name in clusters.index and name not in selected_names:
        selected_names.append(name)

key_clusters = clusters.loc[selected_names].copy()
key_clusters["category"] = "target/reference"
key_clusters.loc[[x for x in estrogen_hits if x in key_clusters.index], "category"] = "estrogen-like"
key_clusters.loc[[x for x in anti_hits if x in key_clusters.index], "category"] = "anti-estrogen"
key_clusters.loc[target_sample, "category"] = "target"
display(key_clusters)


**tips**: t-SNEは`perplexity`が与える影響が大きい。クラスタリング結果そのものと2次元表示を分けて考えたい。


## 6. PCA初期抽出 + varimax回転で潜在因子を作る

Colabでの速度を優先し, ここでは通常の最尤FAではなく, PCAで低次元空間を作った後に varimax 回転 (因子分析手法の一つ)する。

In [ ]:
factor_result = fit_cmap_varimax_factors(
    cmap.compound_by_gene,
    n_components=n_components,
    target_sample=target_sample,
    n_top_genes=n_top_genes,
    random_state=0,
    select_mode="max_abs",
)

print("selected factor:", factor_result.selected_factor)
print(f"{target_sample} score:", round(factor_result.target_score, 3))
display(factor_result.scores.iloc[:5, :8].round(3))


## 7. target-high factorの上位・下位を確認する

選ばれた因子のスコアで化合物をソートする。`target_sample` と同じ側にどのような化合物が来るか, 反対側にどのような化合物が来るかを確認してみよう。


In [ ]:
top_samples = factor_result.ranked_scores.head(top_n)
bottom_samples = factor_result.ranked_scores.tail(top_n).sort_values("factor_score", ascending=True)

print(f"Top {top_n}: high side of {factor_result.selected_factor}")
display(top_samples.round(3))

print(f"Bottom {top_n}: opposite side of {factor_result.selected_factor}")
display(bottom_samples.round(3))

plot_ranked_factor_scores(
    factor_result.ranked_scores,
    top_n=top_n,
    title=f"Samples ranked by {factor_result.selected_factor} score (target={target_sample})",
)
plt.show()


# summary
target_score_table = factor_result.ranked_scores.loc[[target_sample]].copy()
target_score_table["category"] = "target"

reference_scores = summarize_terms(factor_result.ranked_scores)
if len(reference_scores) > 0:
    key_scores = (
        reference_scores
        .drop(index=target_sample, errors="ignore")
        .sort_values("factor_score", ascending=False)
    )
    display(target_score_table.round(3))
    display(key_scores.round(3))
else:
    display(target_score_table.round(3))

**tips**：この結果は因子数や遺伝子選択に依存する。`n_components=10` と `n_components=40` を比較してみよう。

## 9. まとめ

- 遺伝子発現プロファイル使うと, 化合物構造ではなく, 細胞応答の類似性にもとづいて化合物同士の近さを調べることができる。
- 相関ランキングにより, 着目化合物と似た応答を示す化合物, また逆向きの応答を示す化合物を直接確認できる。
- DBSCANは, 似た応答を示す化合物群をざっくり見るための道具になる。ただしクラスタリング結果は距離定義や `eps` などのパラメータに依存し, t-SNE表示はあくまで可視化であるため, 過度な解釈には注意が必要である。
- PCA初期抽出 + varimax回転により, 全体の発現変動を表す低次元空間の中から, 特定の化合物が強く反映される探索的な潜在軸を見つけることができる。
- 回転後の軸の符号は任意である。本実習では, `target_sample` が正のスコアを持つように軸の向きをそろえている。
- 成分数の選択では, 再構成誤差やPPCA-likeなheld-out likelihoodだけでなく, 着目化合物に関係する潜在軸が解釈しやすく得られるかも合わせて考える必要がある。
- 教師なし解析では「数値的に良いモデル」と「薬学的に解釈しやすいモデル」が必ずしも一致しないことに注意する。